# Purpose/risk over the random-100 label set

`PurposeRiskService` loads the `random100-seed20260930-r3` label set that notebook 06
saved, has the store create a run over it, and streams every label through the model.
One row per label, failed ones included, goes to
`data/dsld/runs/random100-seed20260930-r3-<YYMMDDHHMM>-<n>/raw/purpose_risk.jsonl`.

The running log is the controller's and the model's `logging` output, sent to this
notebook: one progress line per finished label with the ETA, a line per retry, and a
warning per label given up on.

In [ ]:
import logging
import sys

import pandas as pd

from engine.repositories.dsld import loader, store
from engine.services.purpose_risk.service import PurposeRiskService

LABEL_SET = "random100-seed20260930-r3"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    stream=sys.stdout,
    force=True,
)

## The label set

What the run will read: 100 cleaned, flattened labels and the `passages_sha` the run records.

In [ ]:
passages_sha, labels = loader.load_flattened(LABEL_SET)
passages_sha, len(labels)

## Is the GPU free?

The model needs ~21 GB. Anything listed under compute apps is holding the card.

In [5]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

memory.free [MiB]
24111 MiB
pid, process_name, used_gpu_memory [MiB]


## The run

The whole set, in its file order. Leaving the `with` block closes the controller,
which stops the server and frees the card -- also on an error or an interrupt. The
last line printed is the run directory and its counts.

In [4]:
with PurposeRiskService() as svc:
    svc.run(LABEL_SET)

2026-09-30 18:58:46,462 INFO engine.controllers.qwen_controller 1/100 2004 tok · 7.2 tok/s · eta 3m13s
2026-09-30 18:58:47,610 INFO engine.controllers.qwen_controller 2/100 3894 tok · 22.0 tok/s · eta 2m32s
2026-09-30 18:58:47,722 INFO engine.controllers.qwen_controller 3/100 5870 tok · 38.7 tok/s · eta 1m44s
2026-09-30 18:58:48,799 INFO engine.controllers.qwen_controller 4/100 7578 tok · 29.4 tok/s · eta 1m43s
2026-09-30 18:58:49,418 INFO engine.controllers.qwen_controller 5/100 9581 tok · 34.9 tok/s · eta 1m33s
2026-09-30 18:58:50,714 INFO engine.controllers.qwen_controller 6/100 11543 tok · 35.0 tok/s · eta 1m37s
2026-09-30 18:58:50,829 INFO engine.controllers.qwen_controller 7/100 13561 tok · 47.9 tok/s · eta 1m24s
2026-09-30 18:58:52,126 INFO engine.controllers.qwen_controller 8/100 15583 tok · 46.7 tok/s · eta 1m28s
2026-09-30 18:58:52,478 INFO engine.controllers.qwen_controller 9/100 17700 tok · 51.5 tok/s · eta 1m20s
2026-09-30 18:58:53,414 INFO engine.controllers.qwen_controll

## The rows

The newest run over the set, one row per label.

In [ ]:
run = max(
    store.RUNS_ROOT.glob(f"{LABEL_SET}-*"),
    key=lambda d: (d.name.rsplit("-", 2)[1], int(d.name.rsplit("-", 1)[1])),
)
print(run)
print(loader.read_manifest(run))

df = pd.DataFrame(loader.read_rows(run))
print(df.status.value_counts().to_string())
df[["dsld_id", "status", "attempts", "failed", "purpose", "risk", "latency_ms"]]

## Run stats

Status and attempts, items per label, the scores the model gave its items, and
tokens and latency per label.

In [7]:
# Status and attempts: how many labels needed a retry, and how many never passed.
print(df.status.value_counts().to_string(), "\n")
print(df.attempts.value_counts().sort_index().rename("labels by attempts").to_string(), "\n")

# How much each label yielded: items per label, and labels with none.
df["n_purpose"] = df.purpose.str.len()
df["n_risk"] = df.risk.str.len()
print(df[["n_purpose", "n_risk"]].describe().round(2).to_string(), "\n")
print(f"no purpose: {(df.n_purpose == 0).sum()}   no risk: {(df.n_risk == 0).sum()}   "
      f"neither: {((df.n_purpose == 0) & (df.n_risk == 0)).sum()}   of {len(df)}\n")

# Scores the model gave its own items.
scores = pd.DataFrame(
    [{"kind": kind, "score": item["score"]} for kind in ("purpose", "risk")
     for items in df[kind] for item in items]
)
print(scores.value_counts().unstack(fill_value=0).to_string(), "\n")

# Tokens and latency per label. Slots run in parallel, so summed latency is not wall time.
print(df[["prompt_tokens", "completion_tokens", "latency_ms"]].describe().round(0).to_string())

status
ok    100 

attempts
1    100 

       n_purpose  n_risk
count     100.00  100.00
mean        1.75    1.92
std         1.97    2.11
min         0.00    0.00
25%         0.00    1.00
50%         1.00    1.00
75%         3.00    2.00
max         9.00   11.00 

no purpose: 35   no risk: 17   neither: 7   of 100

score      5  3  4
kind              
purpose  170  4  1
risk     188  2  2 

       prompt_tokens  completion_tokens  latency_ms
count          100.0              100.0       100.0
mean          1913.0               71.0      3399.0
std            156.0               65.0      2398.0
min           1659.0                2.0       483.0
25%           1783.0               30.0      1682.0
50%           1918.0               54.0      2973.0
75%           1991.0               89.0      4502.0
max           2466.0              313.0     12028.0


## Sample purpose and risk

Each sampled label's passage beside the purpose and risk items the model took from
it, with the model's score for each.

In [8]:
# A seeded handful of labels: the passage the model read, and what it took from it.
# Change SAMPLE_SEED to see others.
SAMPLE_SEED = 0

for row in df.sample(5, random_state=SAMPLE_SEED).itertuples():
    label = labels[row.dsld_id]
    print(f"=== {row.dsld_id}  {label.name}  [{row.status}, {row.attempts} attempt(s)]")
    print(label.passage[:800] + (" ..." if len(label.passage) > 800 else ""))
    print("--- purpose")
    for item in row.purpose:
        print(f"  [{item['score']}] {item['text']}")
    print("--- risk")
    for item in row.risk:
        print(f"  [{item['score']}] {item['text']}")
    print()

=== 22197  Osteo-B Plus  [ok, 1 attempt(s)]
Osteo-B Plus
Sealed with an imprinted safety seal for your protection.
Rev. 8/08
Dietary Supplement
Lot Number: 33985A Exp. Date: APR 12
(C) Copyright 2008
RECOMMENDATION: Three (3) tablets each day as a dietary supplement or as otherwise directed by a healthcare professional.
Caution: Not recommended for pregnant women.
KEEP OUT OF REACH OF CHILDREN
Store in a cool, dry area.
Vitamin D (Form: as Cholecalciferol)
Thiamin (Form: as thiamin mononitrate) (Alt. Name: B1)
Niacin (Form: as Niacinamide)
Vitamin B6 (Form: as Pyridoxine Hydrochloride)
Pantothenic Acid (Form: as Calcium Pantothenate)
Calcium (Form: as Calcium Citrate)
Magnesium (Form: as Magnesium Oxide)
Zinc (Form: as Zinc Gluconate)
Copper (Form: as Copper Gluconate)
Manganese (Form: as Manganese Gluconate)
Boron (Form: as Calcium ...
--- purpose
--- risk
  [5] not recommended for pregnant women

=== 220618  Fo-Ti (He Shou Wu)  [ok, 1 attempt(s)]
Fo-Ti (He Shou Wu)
Fo-Ti (He Shou Wu)

## Most repeated purpose and risk items

In [9]:
# The most repeated items across the run. Repeated risk items are mostly generic
# label warnings; a later filter handles them on the output side.
for kind in ("purpose", "risk"):
    counts = pd.Series([item["text"] for items in df[kind] for item in items]).value_counts()
    print(f"--- {kind}: {counts.sum()} items, {len(counts)} distinct")
    print(counts.head(10).to_string(), "\n")

--- purpose: 175 items, 172 distinct
supports energy production                                                                       2
supports cardiovascular health                                                                   2
promotes bone health                                                                             2
deep nourishment and adaptogenic support                                                         1
supports the digestive process                                                                   1
promotes digestive & whole body health                                                           1
help maintain healthy hydration levels and avoid muscle cramps                                   1
maintain energy, vitality, and a health immune system                                            1
avoid occasional sluggishness & fatigue                                                          1
cordyceps can help fuel natural energy, ignite inner strength and help o

## One failed label in full

The raw answer is kept on every row, so a failure can be read without the GPU.

In [10]:
failed = df[df.status != "ok"]
failed.iloc[0].to_dict() if len(failed) else "no failed labels"

'no failed labels'

## Resume

If the run was interrupted, this reads the run's ids and label set from the run
directory and does only the ids without a row yet. A changed prompt or sampling
setting is refused -- start a new run instead.

In [ ]:
with PurposeRiskService() as svc:
    svc.run(run=run)